# Cargar datos - CustomerChurnX

Este notebook muestra la carga inicial de `Base_de_datos.csv`, valida el esquema minimo esperado y revisa la calidad basica de la informacion antes del EDA y del modelado.

In [1]:
from pathlib import Path
import pandas as pd

candidates = [
    Path.cwd() / "Base_de_datos.csv",
    Path.cwd().parent / "Base_de_datos.csv",
    Path.cwd() / "mlops_pipeline" / "Base_de_datos.csv",
]
DATA_PATH = next((path for path in candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No se encontro Base_de_datos.csv desde el directorio actual")
ROOT = DATA_PATH.parent
DATA_PATH

WindowsPath('C:/Users/Nassi/OneDrive/Desktop/ProyectoM5_Nassim/mlops_pipeline/Base_de_datos.csv')

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")
df.head()

Filas: 5,000
Columnas: 15


,customer_id,signup_month,age,tenure_months,region,channel,plan,sessions_week,avg_session_min,notif_click_rate,support_tickets_3m,discount_pct_3m,late_payments_6m,auto_renew,churn
0,C100000,10,71,49,South,web,Basic,4,10.11,0.318,0,0.138,0,1,0
1,C100001,22,53,44,West,store,Basic,3,6.55,0.329,0,0.000,0,0,0
2,C100002,10,56,51,South,web,Basic,6,12.10,0.120,0,0.001,1,1,0
3,C100003,22,69,48,Center,app,Basic,5,11.01,0.237,0,0.041,0,1,0
4,C100004,20,50,7,South,app,Basic,7,12.97,0.145,0,0.183,0,1,0


## Validacion de esquema

En esta seccion se confirma que la base contiene todas las variables necesarias para el proyecto: identificador del cliente, perfil, comportamiento, pagos y la variable objetivo `churn`.

In [3]:
expected_columns = [
    "customer_id", "signup_month", "age", "tenure_months", "region", "channel",
    "plan", "sessions_week", "avg_session_min", "notif_click_rate",
    "support_tickets_3m", "discount_pct_3m", "late_payments_6m",
    "auto_renew", "churn"
]
missing = sorted(set(expected_columns) - set(df.columns))
extra = sorted(set(df.columns) - set(expected_columns))
print("Columnas faltantes:", missing)
print("Columnas adicionales:", extra)
assert not missing, "La base no cumple el esquema m?nimo requerido"

Columnas faltantes: []
Columnas adicionales: []


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         5000 non-null   str    
 1   signup_month        5000 non-null   int64  
 2   age                 5000 non-null   int64  
 3   tenure_months       5000 non-null   int64  
 4   region              5000 non-null   str    
 5   channel             5000 non-null   str    
 6   plan                5000 non-null   str    
 7   sessions_week       5000 non-null   int64  
 8   avg_session_min     5000 non-null   float64
 9   notif_click_rate    5000 non-null   float64
 10  support_tickets_3m  5000 non-null   int64  
 11  discount_pct_3m     5000 non-null   float64
 12  late_payments_6m    5000 non-null   int64  
 13  auto_renew          5000 non-null   int64  
 14  churn               5000 non-null   int64  
dtypes: float64(3), int64(8), str(4)
memory usage: 685.2 KB


## Revision inicial de calidad

Aqui se revisan tipos, valores nulos, cardinalidad y duplicados. Esta revision sirve para decidir si la base puede pasar al EDA sin un reproceso manual previo.

In [5]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_pct": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique()
})
quality

,dtype,nulos,nulos_pct,unicos
customer_id,str,0,0.0,5000
signup_month,int64,0,0.0,24
age,int64,0,0.0,57
tenure_months,int64,0,0.0,71
region,str,0,0.0,5
channel,str,0,0.0,3
plan,str,0,0.0,3
sessions_week,int64,0,0.0,15
avg_session_min,float64,0,0.0,1591
notif_click_rate,float64,0,0.0,495


In [6]:
duplicate_customers = df["customer_id"].duplicated().sum()
print("Clientes duplicados:", duplicate_customers)
print("Valores objetivo:")
print(df["churn"].value_counts(normalize=True).rename("proporcion"))

Clientes duplicados: 0
Valores objetivo:
churn
0    0.7564
1    0.2436
Name: proporcion, dtype: float64


## Reglas de validacion propuestas

- `customer_id` debe ser unico para evitar clientes repetidos.
- `churn` y `auto_renew` deben ser variables binarias.
- `notif_click_rate` y `discount_pct_3m` deben permanecer entre 0 y 1.
- Variables de conteo como sesiones, tickets, mora y antiguedad no deben ser negativas.
- Variables categoricas como `region`, `channel` y `plan` deben tratarse como texto antes de codificarlas.
- Si estas reglas se cumplen, la base queda lista para exploracion y modelado reproducible.

In [7]:
validation_results = {
    "customer_id_unico": df["customer_id"].is_unique,
    "churn_binario": set(df["churn"].dropna().unique()).issubset({0, 1}),
    "auto_renew_binario": set(df["auto_renew"].dropna().unique()).issubset({0, 1}),
    "notif_click_rate_0_1": df["notif_click_rate"].between(0, 1).all(),
    "discount_pct_3m_0_1": df["discount_pct_3m"].between(0, 1).all(),
    "sin_conteos_negativos": (df[["tenure_months", "sessions_week", "support_tickets_3m", "late_payments_6m"]] >= 0).all().all(),
}
pd.Series(validation_results, name="cumple")

customer_id_unico        True
churn_binario            True
auto_renew_binario       True
notif_click_rate_0_1     True
discount_pct_3m_0_1      True
sin_conteos_negativos    True
Name: cumple, dtype: bool

## Dataset listo para EDA

La base supera las validaciones principales: no presenta nulos, no tiene duplicados por `customer_id` y conserva una estructura consistente. Con esto ya se puede pasar a la etapa de exploracion, ingenieria de variables y entrenamiento.